In [ ]:
# %% 셀 1: 데이터 로드 (텔롭 bbox 예측 모델)
import os, json, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import polars as pl
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed

POS_DIR = "./data/8_telop_position"
STT_DIR = "./data/4_stt_results"
EMB_PATH = "./data/8_text_embeddings.pt"
HEATMAP_DIR = "./data/8_heatmap"
GRID_W = 80
GRID_H = 80
EVAL_PER_CHANNEL = 5
SEED = 42
NUM_WORKERS = 32
FPS = 10
MAX_FRAMES = 2000
MAX_ACTIVE_PER_FRAME = 150
MAX_TEXT_LEN = 200
SLOT_DIM = 5
TIME_DIM = 3

text2emb = torch.load(EMB_PATH, weights_only=True)
EMB_DIM = next(iter(text2emb.values())).shape[0]
ZERO_EMB = torch.zeros(EMB_DIM)
print(f"✅ 임베딩 로드: {len(text2emb):,}개  |  dim={EMB_DIM}")


def stt_frames_to_segments(df_stt):
    rows = df_stt.sort("frame_num").to_dicts()
    segments = []
    cur_text = None
    cur_start_frame = None
    prev_frame = None
    for r in rows:
        t = r["stt_text"]
        f = int(r["frame_num"])
        if t != cur_text:
            if cur_text is not None and cur_text != "":
                segments.append({
                    "start": cur_start_frame / FPS,
                    "end": (prev_frame + 1) / FPS,
                    "text": cur_text.strip(),
                })
            cur_text = t
            cur_start_frame = f
        prev_frame = f
    if cur_text is not None and cur_text != "":
        segments.append({
            "start": cur_start_frame / FPS,
            "end": (prev_frame + 1) / FPS,
            "text": cur_text.strip(),
        })
    return segments


def load_one(args):
    channel, path = args
    with open(path, "r") as f:
        data = json.load(f)
    instances = data.get("instances", [])
    duration = data.get("duration", 0.1)
    if instances:
        duration = max(duration, max(inst["end_sec"] for inst in instances))
    video_name = data.get("video", "")
    file_id = os.path.basename(path)[:-5]
    inst_list = []
    for inst in instances:
        gx = int(np.clip(round(inst["grid_x"]), 0, GRID_W - 1))
        gy = int(np.clip(round(inst["grid_y"]), 0, GRID_H - 1))
        gw = int(np.clip(round(inst["grid_w"]), 1, GRID_W))
        gh = int(np.clip(round(inst["grid_h"]), 1, GRID_H))
        inst_list.append({
            "text": inst["text"],
            "text_len": len(inst["text"]),
            "start": inst["start_sec"],
            "end": inst["end_sec"],
            "x": gx, "y": gy, "w": gw, "h": gh,
        })
    stt_path = os.path.join(STT_DIR, channel, file_id + ".parquet")
    stt_segments = []
    if os.path.exists(stt_path):
        try:
            df_stt = pl.read_parquet(stt_path, glob=False)
            stt_segments = stt_frames_to_segments(df_stt)
        except:
            pass
    return {
        "channel": channel,
        "video_name": video_name,
        "file_id": file_id,
        "instances": inst_list,
        "stt_segments": stt_segments,
        "duration": duration,
    }


json_paths = []
for channel in sorted(os.listdir(POS_DIR)):
    ch_dir = os.path.join(POS_DIR, channel)
    if not os.path.isdir(ch_dir):
        continue
    for fname in sorted(os.listdir(ch_dir)):
        if fname.endswith(".json"):
            json_paths.append((channel, os.path.join(ch_dir, fname)))

samples = []
channel_set = set()
with ProcessPoolExecutor(max_workers=NUM_WORKERS) as pool:
    futures = {pool.submit(load_one, args): args for args in json_paths}
    for fut in tqdm(as_completed(futures), total=len(futures), desc="로드"):
        result = fut.result()
        channel_set.add(result["channel"])
        samples.append(result)

channels = sorted(channel_set)
channel2id = {ch: i for i, ch in enumerate(channels)}

rng = random.Random(SEED)
by_channel = {}
for s in samples:
    if s["channel"] not in by_channel:
        by_channel[s["channel"]] = []
    by_channel[s["channel"]].append(s)

train_samples = []
eval_samples = []
for ch, ch_samples in by_channel.items():
    ch_samples.sort(key=lambda s: s["file_id"])
    rng.shuffle(ch_samples)
    n_eval = min(EVAL_PER_CHANNEL, len(ch_samples))
    eval_samples.extend(ch_samples[:n_eval])
    train_samples.extend(ch_samples[n_eval:])

train_samples = [s for s in train_samples if len(s["instances"]) > 0]
eval_samples_with = [s for s in eval_samples if len(s["instances"]) > 0]

print(f"✅ 채널: {len(channels)}개")
print(f"   train: {len(train_samples):,}  eval: {len(eval_samples_with):,}")

In [ ]:
rng_ch = random.Random(42)
sampled_channels = rng_ch.sample(channels, 67)
sampled_set = set(sampled_channels)

train_samples = [s for s in train_samples if s["channel"] in sampled_set]
eval_samples = [s for s in eval_samples if s["channel"] in sampled_set]
channels = sampled_channels
channel2id = {ch: i for i, ch in enumerate(channels)}

eval_samples_with = [s for s in eval_samples if len(s["instances"]) > 0]
train_samples = [s for s in train_samples if len(s["instances"]) > 0]

print(f"\n🔬 샘플링: {len(channels)}개 채널")
print(f"   train: {len(train_samples):,}  |  eval: {len(eval_samples):,}")

In [ ]:
# %% 셀 2: Dataset + DataLoader (diff vector + cosine scalars + 시간정보)
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

BATCH_SIZE = 8
NUM_WORKERS_DL = 8
MAX_FRAMES = 2000
MAX_ACTIVE_PER_FRAME = 150
MAX_TEXT_LEN = 200
SCALAR_DIM = 8
TIME_DIM = 3
PATCH_SIZE = 8
N_PATCHES_X = GRID_W // PATCH_SIZE
N_PATCHES_Y = GRID_H // PATCH_SIZE
N_PATCHES = N_PATCHES_X * N_PATCHES_Y


class FrameMaskViTDataset(Dataset):
    def __init__(self, samples, channel2id, text2emb):
        self.samples = [s for s in samples if len(s["instances"]) > 0]
        self.channel2id = channel2id
        self.text2emb = text2emb

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        channel_id = self.channel2id[s["channel"]]
        instances = s["instances"]
        duration = max(s["duration"], 0.1)
        n_frames = max(1, min(int(duration * FPS) + 1, MAX_FRAMES))

        n_inst = len(instances)
        inst_starts = np.array([inst["start"] for inst in instances])
        inst_ends   = np.array([inst["end"]   for inst in instances])
        inst_tlens  = np.array([inst["text_len"] for inst in instances])
        inst_cx = np.array([inst["x"] for inst in instances])
        inst_cy = np.array([inst["y"] for inst in instances])
        inst_w  = np.array([inst["w"] for inst in instances])
        inst_h  = np.array([inst["h"] for inst in instances])

        inst_x0 = np.maximum(0, inst_cx - inst_w // 2)
        inst_y0 = np.maximum(0, inst_cy - inst_h // 2)
        inst_x1 = np.minimum(GRID_W, inst_x0 + inst_w)
        inst_y1 = np.minimum(GRID_H, inst_y0 + inst_h)

        channel_emb = F.normalize(self.text2emb.get(s["channel"], ZERO_EMB), dim=-1)
        video_emb   = F.normalize(self.text2emb.get(s["video_name"], ZERO_EMB), dim=-1)
        inst_embs   = torch.stack([
            self.text2emb.get(inst["text"], ZERO_EMB) for inst in instances
        ])
        inst_embs = F.normalize(inst_embs, dim=-1)

        inst_diff_ch  = inst_embs - channel_emb.unsqueeze(0)
        inst_diff_vid = inst_embs - video_emb.unsqueeze(0)
        inst_diff_stt = torch.zeros(n_inst, EMB_DIM)

        ch_sims  = F.cosine_similarity(inst_embs, channel_emb.unsqueeze(0), dim=-1).numpy()
        vid_sims = F.cosine_similarity(inst_embs, video_emb.unsqueeze(0), dim=-1).numpy()

        stt_sims = np.zeros(n_inst, dtype=np.float32)
        has_stts = np.zeros(n_inst, dtype=np.float32)
        stt_segments = s["stt_segments"]
        if len(stt_segments) > 0:
            stt_starts = np.array([seg["start"] for seg in stt_segments])
            stt_ends   = np.array([seg["end"]   for seg in stt_segments])
            stt_embs_raw = torch.stack([
                self.text2emb.get(seg["text"], ZERO_EMB) for seg in stt_segments
            ])
            stt_embs = F.normalize(stt_embs_raw, dim=-1)
            for i in range(n_inst):
                mid = (inst_starts[i] + inst_ends[i]) / 2
                stt_active = (stt_starts <= mid) & (stt_ends > mid)
                stt_active_idx = np.where(stt_active)[0]
                if len(stt_active_idx) > 0:
                    inst_diff_stt[i] = inst_embs[i] - stt_embs[stt_active_idx[0]]
                    stt_sims[i] = F.cosine_similarity(
                        inst_embs[i].unsqueeze(0),
                        stt_embs[stt_active_idx[0]].unsqueeze(0),
                    ).item()
                    has_stts[i] = 1.0

        inst_scalars = np.zeros((n_inst, SCALAR_DIM), dtype=np.float32)
        inst_scalars[:, 0] = inst_tlens / MAX_TEXT_LEN
        inst_scalars[:, 1] = ch_sims
        inst_scalars[:, 2] = vid_sims
        inst_scalars[:, 3] = stt_sims
        inst_scalars[:, 4] = has_stts
        inst_scalars[:, 5] = inst_starts / max(duration, 0.1)
        inst_scalars[:, 6] = inst_ends / max(duration, 0.1)
        inst_scalars[:, 7] = (inst_ends - inst_starts) / max(duration, 0.1)

        times = np.arange(n_frames, dtype=np.float32) / FPS
        active_matrix = (
            (inst_starts[None, :] <= times[:, None] + 0.05) &
            (inst_ends[None, :]   >  times[:, None])
        )

        telop_mask    = np.zeros((n_frames, MAX_ACTIVE_PER_FRAME), dtype=np.bool_)
        slot_bbox     = np.zeros((n_frames, MAX_ACTIVE_PER_FRAME, 4), dtype=np.int16)
        slot_inst_idx = np.full((n_frames, MAX_ACTIVE_PER_FRAME), -1, dtype=np.int64)

        for fi in range(n_frames):
            active_idx = np.where(active_matrix[fi])[0]
            if len(active_idx) > 0:
                sorted_order = np.argsort(inst_tlens[active_idx])[::-1][:MAX_ACTIVE_PER_FRAME]
                sorted_idx = active_idx[sorted_order]
                for si, ai in enumerate(sorted_idx):
                    telop_mask[fi, si] = True
                    slot_bbox[fi, si] = [inst_x0[ai], inst_y0[ai],
                                         inst_x1[ai], inst_y1[ai]]
                    slot_inst_idx[fi, si] = ai

        time_feats = np.zeros((n_frames, TIME_DIM), dtype=np.float32)
        t_norm = times / max(duration, 1.0)
        time_feats[:, 0] = t_norm
        time_feats[:, 1] = np.sin(2 * np.pi * t_norm)
        time_feats[:, 2] = np.cos(2 * np.pi * t_norm)

        return {
            "channel_id":    torch.tensor(channel_id, dtype=torch.long),
            "telop_mask":    torch.from_numpy(telop_mask),
            "time_feats":    torch.from_numpy(time_feats),
            "slot_bbox":     torch.from_numpy(slot_bbox.astype(np.int32)),
            "slot_inst_idx": torch.from_numpy(slot_inst_idx),
            "inst_diff_ch":  inst_diff_ch,
            "inst_diff_vid": inst_diff_vid,
            "inst_diff_stt": inst_diff_stt,
            "inst_scalars":  torch.from_numpy(inst_scalars),
            "n_frames":      n_frames,
        }


def collate_fn(batch):
    max_T = max(b["n_frames"] for b in batch)
    max_inst = max(b["inst_diff_ch"].shape[0] for b in batch)
    B = len(batch)

    channel_ids   = torch.stack([b["channel_id"] for b in batch])
    telop_mask    = torch.zeros(B, max_T, MAX_ACTIVE_PER_FRAME, dtype=torch.bool)
    time_feats    = torch.zeros(B, max_T, TIME_DIM)
    slot_bbox     = torch.zeros(B, max_T, MAX_ACTIVE_PER_FRAME, 4, dtype=torch.int32)
    frame_mask    = torch.zeros(B, max_T, dtype=torch.bool)
    slot_inst_idx = torch.full((B, max_T, MAX_ACTIVE_PER_FRAME), -1, dtype=torch.long)
    inst_diff_ch  = torch.zeros(B, max_inst, EMB_DIM)
    inst_diff_vid = torch.zeros(B, max_inst, EMB_DIM)
    inst_diff_stt = torch.zeros(B, max_inst, EMB_DIM)
    inst_scalars  = torch.zeros(B, max_inst, SCALAR_DIM)

    for i, b in enumerate(batch):
        T = b["n_frames"]
        n_inst = b["inst_diff_ch"].shape[0]
        telop_mask[i, :T]    = b["telop_mask"]
        time_feats[i, :T]    = b["time_feats"]
        slot_bbox[i, :T]     = b["slot_bbox"]
        frame_mask[i, :T]    = True
        slot_inst_idx[i, :T] = b["slot_inst_idx"]
        inst_diff_ch[i, :n_inst]  = b["inst_diff_ch"]
        inst_diff_vid[i, :n_inst] = b["inst_diff_vid"]
        inst_diff_stt[i, :n_inst] = b["inst_diff_stt"]
        inst_scalars[i, :n_inst]  = b["inst_scalars"]

    return {
        "channel_ids":   channel_ids,
        "telop_mask":    telop_mask,
        "time_feats":    time_feats,
        "slot_bbox":     slot_bbox,
        "frame_mask":    frame_mask,
        "slot_inst_idx": slot_inst_idx,
        "inst_diff_ch":  inst_diff_ch,
        "inst_diff_vid": inst_diff_vid,
        "inst_diff_stt": inst_diff_stt,
        "inst_scalars":  inst_scalars,
    }


train_ds = FrameMaskViTDataset(train_samples, channel2id, text2emb)
eval_ds  = FrameMaskViTDataset(eval_samples, channel2id, text2emb)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS_DL, pin_memory=True,
    collate_fn=collate_fn, drop_last=True,
    persistent_workers=True,
)
eval_loader = DataLoader(
    eval_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS_DL, pin_memory=True,
    collate_fn=collate_fn,
    persistent_workers=True,
)

print(f"✅ Dataset: train={len(train_ds):,}  eval={len(eval_ds):,}")
print(f"   EMB_DIM={EMB_DIM}  SCALAR_DIM={SCALAR_DIM}")

batch = next(iter(train_loader))
print(f"\n✅ 배치 확인")
for k, v in batch.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k}: {v.shape}")

In [ ]:
# %% 셀 3: 모델 (Deformable DETR: inst self-attn 4L + temporal 4L + spatial 4L + deformable decoder 6L)
from torch.utils.checkpoint import checkpoint as ckpt_fn
import math

D_MODEL = 256
N_HEADS = 8
N_POINTS = 4
N_LAYERS_TEMPORAL = 4
N_LAYERS_SPATIAL = 4
N_LAYERS_DECODER = 6
N_LAYERS_INST = 4
D_FF = 512
DROPOUT = 0.1
INTRA_CHUNK = 512
SPATIAL_CHUNK = 512
SLOT_DECODE_CHUNK = 512


class IntraFrameModule(nn.Module):
    def __init__(self, d_model=D_MODEL):
        super().__init__()
        self.pool_query = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.combine = nn.Sequential(
            nn.Linear(d_model * 3 + 1, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )
        self.norm = nn.LayerNorm(d_model)

    def forward(self, tokens, slot_mask):
        N, K, D = tokens.shape
        dtype = tokens.dtype
        d = D

        slot_tokens = tokens * slot_mask.unsqueeze(-1).float()

        slot_pad = ~slot_mask
        count = slot_mask.sum(dim=1, keepdim=True).float().clamp(min=1)
        count_norm = count / MAX_ACTIVE_PER_FRAME

        masked_tokens = tokens.masked_fill(slot_pad.unsqueeze(-1), 0.0)
        mean_pool = masked_tokens.sum(dim=1) / count

        masked_for_max = tokens.masked_fill(slot_pad.unsqueeze(-1), float("-inf"))
        max_pool = masked_for_max.max(dim=1).values
        all_pad = ~slot_mask.any(dim=1, keepdim=True)
        max_pool = max_pool.masked_fill(all_pad, 0.0)

        q = self.pool_query.expand(N, -1, -1)
        a = torch.bmm(q, tokens.transpose(1, 2)) / (d ** 0.5)
        a = a.masked_fill(slot_pad.unsqueeze(1), float("-inf"))
        a = F.softmax(a, dim=-1)
        a = a.masked_fill(a.isnan(), 0.0)
        attn_pool = torch.bmm(a, tokens).squeeze(1)

        combined = torch.cat([attn_pool, mean_pool, max_pool, count_norm], dim=-1)
        pooled = self.norm(self.combine(combined))
        pooled = pooled.masked_fill(all_pad, 0.0)

        return pooled.to(dtype), slot_tokens.to(dtype)


class DeformableCrossAttention(nn.Module):
    """Single-scale deformable cross-attention.

    Query [N, K, D] attends to value [N, H*W, D] by sampling
    n_heads * n_points locations around per-query reference points.
    """
    def __init__(self, d_model=D_MODEL, n_heads=N_HEADS, n_points=N_POINTS):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.n_points = n_points
        self.head_dim = d_model // n_heads

        self.sampling_offsets = nn.Linear(d_model, n_heads * n_points * 2)
        self.attention_weights = nn.Linear(d_model, n_heads * n_points)
        self.value_proj = nn.Linear(d_model, d_model)
        self.output_proj = nn.Linear(d_model, d_model)

        self._reset_parameters()

    def _reset_parameters(self):
        nn.init.constant_(self.sampling_offsets.weight, 0.0)
        thetas = torch.arange(self.n_heads, dtype=torch.float32) * (2.0 * math.pi / self.n_heads)
        grid_init = torch.stack([thetas.cos(), thetas.sin()], -1)
        grid_init = grid_init / grid_init.abs().max(-1, keepdim=True)[0]
        grid_init = grid_init.reshape(self.n_heads, 1, 2).repeat(1, self.n_points, 1)
        for i in range(self.n_points):
            grid_init[:, i, :] *= (i + 1)
        with torch.no_grad():
            self.sampling_offsets.bias = nn.Parameter(grid_init.reshape(-1))
        nn.init.constant_(self.attention_weights.weight, 0.0)
        nn.init.constant_(self.attention_weights.bias, 0.0)
        nn.init.xavier_uniform_(self.value_proj.weight)
        nn.init.constant_(self.value_proj.bias, 0.0)
        nn.init.xavier_uniform_(self.output_proj.weight)
        nn.init.constant_(self.output_proj.bias, 0.0)

    def forward(self, query, value, reference_points, spatial_shape):
        N, K, D = query.shape
        H, W = spatial_shape

        v = self.value_proj(value)
        v = v.reshape(N, H * W, self.n_heads, self.head_dim)
        v = v.permute(0, 2, 3, 1).reshape(N * self.n_heads, self.head_dim, H, W)

        offsets = self.sampling_offsets(query).reshape(
            N, K, self.n_heads, self.n_points, 2
        )
        norm = torch.tensor([W, H], dtype=offsets.dtype, device=offsets.device)
        offsets_norm = offsets / norm.reshape(1, 1, 1, 1, 2)

        ref = reference_points.reshape(N, K, 1, 1, 2)
        sampling_loc = ref + offsets_norm
        grid = 2.0 * sampling_loc - 1.0
        grid = grid.permute(0, 2, 1, 3, 4).reshape(
            N * self.n_heads, K, self.n_points, 2
        )

        sampled = F.grid_sample(
            v, grid.to(v.dtype), mode="bilinear",
            padding_mode="zeros", align_corners=False,
        )
        sampled = sampled.reshape(N, self.n_heads, self.head_dim, K, self.n_points)
        sampled = sampled.permute(0, 3, 1, 4, 2)

        weights = self.attention_weights(query).reshape(
            N, K, self.n_heads, self.n_points
        )
        weights = F.softmax(weights, dim=-1)

        out = (sampled * weights.unsqueeze(-1)).sum(dim=3)
        out = out.reshape(N, K, D)
        return self.output_proj(out)


class SlotXYDecoderLayer(nn.Module):
    def __init__(self, d_model=D_MODEL, n_heads=N_HEADS, n_points=N_POINTS,
                 d_ff=D_FF, dropout=DROPOUT):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.dropout = dropout

        self.sa_qkv = nn.Linear(d_model, d_model * 3)
        self.sa_out = nn.Linear(d_model, d_model)
        self.self_norm = nn.LayerNorm(d_model)
        self.self_ffn = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_ff, d_model), nn.Dropout(dropout))
        self.self_ffn_norm = nn.LayerNorm(d_model)

        self.def_attn = DeformableCrossAttention(d_model, n_heads, n_points)
        self.cross_norm = nn.LayerNorm(d_model)
        self.cross_ffn = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_ff, d_model), nn.Dropout(dropout))
        self.cross_ffn_norm = nn.LayerNorm(d_model)

    def _reshape(self, x):
        B, S, _ = x.shape
        return x.reshape(B, S, self.n_heads, self.head_dim).transpose(1, 2)

    def forward(self, q, patch_features, slot_pad, reference_points, spatial_shape):
        B, S, D = q.shape

        valid = (~slot_pad).unsqueeze(-1).float()
        q_masked = q * valid

        qkv = self.sa_qkv(q_masked)
        sq, sk, sv = qkv.chunk(3, dim=-1)
        sq, sk, sv = self._reshape(sq), self._reshape(sk), self._reshape(sv)

        attn_mask = torch.zeros(B, 1, 1, S, device=q.device, dtype=q.dtype)
        attn_mask = attn_mask.masked_fill(slot_pad[:, None, None, :], float("-inf"))

        sa_out = F.scaled_dot_product_attention(
            sq, sk, sv, attn_mask=attn_mask,
            dropout_p=self.dropout if self.training else 0.0)
        sa_out = sa_out.transpose(1, 2).reshape(B, S, D)
        sa_out = self.sa_out(sa_out)
        q = self.self_norm(q_masked + sa_out)
        q = self.self_ffn_norm(q + self.self_ffn(q))
        q = q * valid

        ca_out = self.def_attn(q, patch_features, reference_points, spatial_shape)
        q = self.cross_norm(q + ca_out)
        q = self.cross_ffn_norm(q + self.cross_ffn(q))
        q = q * valid

        return q


class SlotXYDecoder(nn.Module):
    def __init__(self, d_model=D_MODEL, n_heads=N_HEADS, n_points=N_POINTS,
                 d_ff=D_FF, dropout=DROPOUT, n_layers=N_LAYERS_DECODER):
        super().__init__()
        self.query_proj = nn.Sequential(
            nn.Linear(d_model * 2, d_model), nn.GELU(), nn.Linear(d_model, d_model))
        self.query_norm = nn.LayerNorm(d_model)

        self.reference_points = nn.Linear(d_model, 2)
        nn.init.xavier_uniform_(self.reference_points.weight, gain=1.0)
        nn.init.constant_(self.reference_points.bias, 0.0)

        self.layers = nn.ModuleList([
            SlotXYDecoderLayer(d_model, n_heads, n_points, d_ff, dropout)
            for _ in range(n_layers)
        ])

        self.x_head = nn.Sequential(
            nn.Linear(d_model, d_model), nn.GELU(), nn.Linear(d_model, GRID_W))
        self.y_head = nn.Sequential(
            nn.Linear(d_model, d_model), nn.GELU(), nn.Linear(d_model, GRID_H))

    def forward(self, patch_features, slot_query_input, slot_mask, spatial_shape):
        q = self.query_norm(self.query_proj(slot_query_input))
        slot_pad = ~slot_mask

        reference_points = torch.sigmoid(self.reference_points(q))

        for layer in self.layers:
            q = ckpt_fn(layer, q, patch_features, slot_pad,
                        reference_points, spatial_shape, use_reentrant=False)

        return self.x_head(q), self.y_head(q)


class ViTPatchMaskModel(nn.Module):
    def __init__(self, n_channels, emb_dim=EMB_DIM, d_model=D_MODEL, n_heads=N_HEADS,
                 n_layers_temporal=N_LAYERS_TEMPORAL,
                 n_layers_spatial=N_LAYERS_SPATIAL,
                 d_ff=D_FF, dropout=DROPOUT):
        super().__init__()

        self.ch_proj = nn.Sequential(
            nn.Linear(emb_dim + 1, d_model // 2), nn.GELU())
        self.vid_proj = nn.Sequential(
            nn.Linear(emb_dim + 1, d_model // 2), nn.GELU())
        self.stt_proj = nn.Sequential(
            nn.Linear(emb_dim + 2, d_model // 2), nn.GELU())
        self.len_proj = nn.Sequential(
            nn.Linear(4, d_model // 2), nn.GELU())
        self.slot_combine = nn.Sequential(
            nn.Linear(d_model * 2, d_model), nn.GELU(), nn.Linear(d_model, d_model))

        inst_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True, activation="gelu")
        self.inst_self_attn = nn.TransformerEncoder(
            inst_layer, num_layers=N_LAYERS_INST, enable_nested_tensor=False)

        self.slot_pos_emb = nn.Embedding(MAX_ACTIVE_PER_FRAME, d_model)
        nn.init.normal_(self.slot_pos_emb.weight, mean=0.0, std=0.02)

        self.intra_frame = IntraFrameModule(d_model)

        self.channel_emb = nn.Embedding(n_channels, d_model)
        self.time_proj = nn.Sequential(
            nn.Linear(TIME_DIM, 64), nn.GELU(), nn.Linear(64, d_model))
        self.temporal_norm = nn.LayerNorm(d_model)

        temporal_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True, activation="gelu")
        self.temporal_transformer = nn.TransformerEncoder(
            temporal_layer, num_layers=n_layers_temporal, enable_nested_tensor=False)

        self.patch_pos_emb = nn.Parameter(torch.randn(1, N_PATCHES, d_model) * 0.02)
        self.spatial_norm = nn.LayerNorm(d_model)

        spatial_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True, activation="gelu")
        self.spatial_transformer = nn.TransformerEncoder(
            spatial_layer, num_layers=n_layers_spatial, enable_nested_tensor=False)

        self.slot_decoder = SlotXYDecoder(d_model, n_heads, N_POINTS, d_ff, dropout)
        self.spatial_shape = (N_PATCHES_Y, N_PATCHES_X)

    def forward(self, channel_ids, telop_mask, time_feats, frame_mask,
                slot_inst_idx, inst_diff_ch, inst_diff_vid, inst_diff_stt, inst_scalars):
        B, T, K = telop_mask.shape
        device = telop_mask.device
        dtype = inst_diff_ch.dtype
        BT = B * T
        D = D_MODEL

        ch_input = torch.cat([inst_diff_ch, inst_scalars[..., 1:2]], dim=-1)
        vid_input = torch.cat([inst_diff_vid, inst_scalars[..., 2:3]], dim=-1)
        stt_input = torch.cat([inst_diff_stt,
                               inst_scalars[..., 3:4],
                               inst_scalars[..., 4:5]], dim=-1)
        len_input = torch.cat([
            inst_scalars[..., 0:1],
            inst_scalars[..., 5:8],
        ], dim=-1)

        proj_ch  = self.ch_proj(ch_input)
        proj_vid = self.vid_proj(vid_input)
        proj_stt = self.stt_proj(stt_input)
        proj_len = self.len_proj(len_input)

        inst_tokens = self.slot_combine(
            torch.cat([proj_ch, proj_vid, proj_stt, proj_len], dim=-1)
        )

        inst_mask = (inst_scalars[..., 0] != 0)
        inst_pad = ~inst_mask

        inst_tokens = self.inst_self_attn(
            inst_tokens, src_key_padding_mask=inst_pad)

        smask_flat = telop_mask.reshape(BT, K)
        idx_flat = slot_inst_idx.reshape(BT, K).clamp(min=0)
        batch_for_bt = torch.arange(B, device=device).unsqueeze(1).expand(-1, T).reshape(BT)

        slot_idx = torch.arange(K, device=device)
        pos_emb = self.slot_pos_emb(slot_idx).unsqueeze(0)

        pooled_list, slot_tok_list = [], []
        for start in range(0, BT, INTRA_CHUNK):
            end = min(start + INTRA_CHUNK, BT)
            chunk_bidx = batch_for_bt[start:end].unsqueeze(1).expand(-1, K)
            chunk_iidx = idx_flat[start:end]
            chunk_tokens = inst_tokens[chunk_bidx, chunk_iidx] + pos_emb
            p, s = self.intra_frame(chunk_tokens, smask_flat[start:end])
            pooled_list.append(p)
            slot_tok_list.append(s)

        frame_tokens = torch.cat(pooled_list, dim=0).reshape(B, T, D)
        all_slot_tokens = torch.cat(slot_tok_list, dim=0).reshape(B, T, K, D)

        ch = self.channel_emb(channel_ids).unsqueeze(1).expand(-1, T, -1)
        time = self.time_proj(time_feats)
        tokens = self.temporal_norm(frame_tokens + ch + time)

        temporal_out = self.temporal_transformer(
            tokens, src_key_padding_mask=~frame_mask)

        temporal_flat = temporal_out.reshape(BT, D)
        slot_tok_flat = all_slot_tokens.reshape(BT, K, D)
        valid_flat = frame_mask.reshape(BT) & smask_flat.any(dim=1)
        valid_idx = valid_flat.nonzero(as_tuple=True)[0]
        n_valid = valid_idx.shape[0]

        all_x = torch.zeros(BT, K, GRID_W, device=device, dtype=dtype)
        all_y = torch.zeros(BT, K, GRID_H, device=device, dtype=dtype)

        if n_valid > 0:
            valid_ctx = temporal_flat[valid_idx]
            valid_slot = slot_tok_flat[valid_idx]
            valid_smask = smask_flat[valid_idx]

            queries = valid_ctx.unsqueeze(1) + self.patch_pos_emb
            queries = self.spatial_norm(queries)

            patch_feat_list = []
            for start in range(0, n_valid, SPATIAL_CHUNK):
                end = min(start + SPATIAL_CHUNK, n_valid)
                spatial_out = self.spatial_transformer(queries[start:end])
                patch_feat_list.append(spatial_out)

            patch_features = torch.cat(patch_feat_list, dim=0)

            ctx_expand = valid_ctx.unsqueeze(1).expand(-1, K, -1)
            slot_query_input = torch.cat([ctx_expand, valid_slot], dim=-1)

            x_list, y_list = [], []
            for start in range(0, n_valid, SLOT_DECODE_CHUNK):
                end = min(start + SLOT_DECODE_CHUNK, n_valid)
                xo, yo = self.slot_decoder(
                    patch_features[start:end],
                    slot_query_input[start:end],
                    valid_smask[start:end],
                    self.spatial_shape,
                )
                x_list.append(xo)
                y_list.append(yo)

            all_x[valid_idx] = torch.cat(x_list, dim=0).to(dtype)
            all_y[valid_idx] = torch.cat(y_list, dim=0).to(dtype)

        return all_x.reshape(B, T, K, GRID_W), all_y.reshape(B, T, K, GRID_H)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ViTPatchMaskModel(n_channels=len(channels)).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"🖥️  Device: {device}")
print(f"📐 파라미터: {n_params:,}")
print(f"   inst self-attn: {N_LAYERS_INST}L")
print(f"   temporal: {N_LAYERS_TEMPORAL}L  spatial: {N_LAYERS_SPATIAL}L")
print(f"   decoder: {N_LAYERS_DECODER}L (SDPA self-attn + deformable cross-attn, K={N_POINTS}pts)")


In [ ]:
# %% 셀 4: 학습 (focal + 2D threshold)
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from scipy.optimize import linear_sum_assignment

EPOCHS = 50
LR = 1e-4
FOCAL_ALPHA_X = 2.7 / (1 + 2.7)
FOCAL_ALPHA_Y = 25.6 / (1 + 25.6)
FOCAL_GAMMA = 2.0
SAVE_DIR = "./model/8_layout_slot_xy_deformable"
os.makedirs(SAVE_DIR, exist_ok=True)

_x_range = torch.arange(GRID_W, device=device).unsqueeze(0)
_y_range = torch.arange(GRID_H, device=device).unsqueeze(0)


def focal_loss_1d(logits, targets, alpha, gamma=FOCAL_GAMMA):
    bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    probs = torch.sigmoid(logits)
    p_t = probs * targets + (1 - probs) * (1 - targets)
    alpha_t = alpha * targets + (1 - alpha) * (1 - targets)
    focal_weight = alpha_t * (1 - p_t) ** gamma
    return (focal_weight * bce).mean()


def slot_xy_loss(x_logits, y_logits, slot_bbox, active_mask):
    if not active_mask.any():
        return x_logits.sum() * 0.0

    active_x = x_logits[active_mask]
    active_y = y_logits[active_mask]

    bbox = slot_bbox[active_mask].long()
    x0, y0, x1, y1 = bbox[:, 0], bbox[:, 1], bbox[:, 2], bbox[:, 3]
    gt_x = ((_x_range >= x0.unsqueeze(1)) & (_x_range < x1.unsqueeze(1))).float()
    gt_y = ((_y_range >= y0.unsqueeze(1)) & (_y_range < y1.unsqueeze(1))).float()

    loss_x = focal_loss_1d(active_x, gt_x, alpha=FOCAL_ALPHA_X)
    loss_y = focal_loss_1d(active_y, gt_y, alpha=FOCAL_ALPHA_Y)

    return loss_x + loss_y


@torch.no_grad()
def compute_slot_metrics(x_logits, y_logits, telop_mask, slot_bbox, frame_mask):
    active_mask = telop_mask & frame_mask.unsqueeze(-1)

    if not active_mask.any():
        return None

    active_x = torch.sigmoid(x_logits[active_mask])
    active_y = torch.sigmoid(y_logits[active_mask])

    bbox = slot_bbox[active_mask].long()
    x0, y0, x1, y1 = bbox[:, 0], bbox[:, 1], bbox[:, 2], bbox[:, 3]
    gt_x_bool = ((_x_range >= x0.unsqueeze(1)) & (_x_range < x1.unsqueeze(1)))
    gt_y_bool = ((_y_range >= y0.unsqueeze(1)) & (_y_range < y1.unsqueeze(1)))

    th_list = [i / 10 for i in range(1, 10)]

    best_xf1 = 0.0
    for th in th_list:
        xb = active_x > th
        tp = (xb & gt_x_bool).sum().item()
        fp = (xb & ~gt_x_bool).sum().item()
        fn = (~xb & gt_x_bool).sum().item()
        p = tp / (tp + fp + 1e-8)
        r = tp / (tp + fn + 1e-8)
        f1 = 2 * p * r / (p + r + 1e-8)
        if f1 > best_xf1:
            best_xf1 = f1

    best_yf1 = 0.0
    for th in th_list:
        yb = active_y > th
        tp = (yb & gt_y_bool).sum().item()
        fp = (yb & ~gt_y_bool).sum().item()
        fn = (~yb & gt_y_bool).sum().item()
        p = tp / (tp + fp + 1e-8)
        r = tp / (tp + fn + 1e-8)
        f1 = 2 * p * r / (p + r + 1e-8)
        if f1 > best_yf1:
            best_yf1 = f1

    gt_x_sum = gt_x_bool.sum(dim=-1).float()
    gt_y_sum = gt_y_bool.sum(dim=-1).float()
    gt_area = gt_x_sum * gt_y_sum
    gt_total = gt_area.sum().item()

    best_f1, best_p, best_r = 0.0, 0.0, 0.0
    best_xth, best_yth = 0.5, 0.5

    coarse = [i / 10 for i in range(1, 10)]
    for thx in coarse:
        x_bin = active_x > thx
        x_pred_sum = x_bin.sum(dim=-1).float()
        x_match = (x_bin & gt_x_bool).sum(dim=-1).float()
        for thy in coarse:
            y_bin = active_y > thy
            y_pred_sum = y_bin.sum(dim=-1).float()
            y_match = (y_bin & gt_y_bool).sum(dim=-1).float()
            tp = (x_match * y_match).sum().item()
            pred_area = (x_pred_sum * y_pred_sum).sum().item()
            fp = pred_area - tp
            fn = gt_total - tp
            p = tp / (tp + fp + 1e-8)
            r = tp / (tp + fn + 1e-8)
            f1 = 2 * p * r / (p + r + 1e-8)
            if f1 > best_f1:
                best_f1, best_p, best_r = f1, p, r
                best_xth, best_yth = thx, thy

    fine_x = [best_xth + d / 100 for d in range(-9, 10)]
    fine_y = [best_yth + d / 100 for d in range(-9, 10)]
    fine_x = [t for t in fine_x if 0.01 <= t <= 0.99]
    fine_y = [t for t in fine_y if 0.01 <= t <= 0.99]

    for thx in fine_x:
        x_bin = active_x > thx
        x_pred_sum = x_bin.sum(dim=-1).float()
        x_match = (x_bin & gt_x_bool).sum(dim=-1).float()
        for thy in fine_y:
            y_bin = active_y > thy
            y_pred_sum = y_bin.sum(dim=-1).float()
            y_match = (y_bin & gt_y_bool).sum(dim=-1).float()
            tp = (x_match * y_match).sum().item()
            pred_area = (x_pred_sum * y_pred_sum).sum().item()
            fp = pred_area - tp
            fn = gt_total - tp
            p = tp / (tp + fp + 1e-8)
            r = tp / (tp + fn + 1e-8)
            f1 = 2 * p * r / (p + r + 1e-8)
            if f1 > best_f1:
                best_f1, best_p, best_r = f1, p, r
                best_xth, best_yth = thx, thy

    result = {
        "P": best_p, "R": best_r, "F1": best_f1,
        "bestF1": best_f1,
        "bestThX": best_xth, "bestThY": best_yth,
        "xF1": best_xf1,
        "yF1": best_yf1,
    }

    B, T, K = active_mask.shape
    x_prob = torch.sigmoid(x_logits)
    y_prob = torch.sigmoid(y_logits)

    has_frame = active_mask.any(dim=2)
    b_idx, t_idx = has_frame.nonzero(as_tuple=True)
    n_total = len(b_idx)
    if n_total > 500:
        sample_idx = torch.randperm(n_total, device=x_logits.device)[:500]
        b_idx = b_idx[sample_idx]
        t_idx = t_idx[sample_idx]

    hung_tp, hung_fp, hung_fn = 0, 0, 0
    for bi, ti in zip(b_idx.tolist(), t_idx.tolist()):
        am = active_mask[bi, ti]
        active_slots = am.nonzero(as_tuple=True)[0]
        n_active = len(active_slots)
        if n_active == 0:
            continue

        xp = x_prob[bi, ti, active_slots]
        yp = y_prob[bi, ti, active_slots]
        bb = slot_bbox[bi, ti, active_slots].long()

        pred_masks = (yp > best_yth).unsqueeze(-1) & (xp > best_xth).unsqueeze(-2)
        gt_masks = torch.zeros(n_active, GRID_H, GRID_W, device=x_logits.device, dtype=torch.bool)
        for j in range(n_active):
            gt_masks[j, bb[j, 1]:bb[j, 3], bb[j, 0]:bb[j, 2]] = True

        inter = (pred_masks.unsqueeze(1) & gt_masks.unsqueeze(0)).sum(dim=(-1, -2)).float()
        union = (pred_masks.unsqueeze(1) | gt_masks.unsqueeze(0)).sum(dim=(-1, -2)).float()
        iou_matrix = inter / (union + 1e-8)

        cost = (1 - iou_matrix).cpu().numpy()
        row_ind, col_ind = linear_sum_assignment(cost)

        for ri, ci in zip(row_ind, col_ind):
            pm = pred_masks[ri]
            gm = gt_masks[ci]
            hung_tp += (pm & gm).sum().item()
            hung_fp += (pm & ~gm).sum().item()
            hung_fn += (~pm & gm).sum().item()

    hung_p = hung_tp / (hung_tp + hung_fp + 1e-8)
    hung_r = hung_tp / (hung_tp + hung_fn + 1e-8)
    result["hungF1"] = 2 * hung_p * hung_r / (hung_p + hung_r + 1e-8)

    b_idx2, t_idx2 = has_frame.nonzero(as_tuple=True)
    n_total2 = len(b_idx2)
    if n_total2 > 2000:
        sample_idx = torch.randperm(n_total2, device=x_logits.device)[:2000]
        b_idx2 = b_idx2[sample_idx]
        t_idx2 = t_idx2[sample_idx]

    merged_preds, merged_gts = [], []
    for bi, ti in zip(b_idx2.tolist(), t_idx2.tolist()):
        am = active_mask[bi, ti]
        bb = slot_bbox[bi, ti]

        xp = x_prob[bi, ti, am]
        yp = y_prob[bi, ti, am]
        pred_x_bin = xp > best_xth
        pred_y_bin = yp > best_yth
        slot_2d = pred_y_bin.unsqueeze(-1) & pred_x_bin.unsqueeze(-2)
        pred_merged = slot_2d.any(dim=0)

        gt_boxes = bb[am].long()
        gt_merged = torch.zeros(GRID_H, GRID_W, device=x_logits.device, dtype=torch.bool)
        for box in gt_boxes:
            gt_merged[box[1]:box[3], box[0]:box[2]] = True

        merged_preds.append(pred_merged)
        merged_gts.append(gt_merged)

    mp = torch.stack(merged_preds).reshape(-1)
    mg = torch.stack(merged_gts).reshape(-1)

    tp = (mp & mg).sum().item()
    fp = (mp & ~mg).sum().item()
    fn = (~mp & mg).sum().item()
    p_m = tp / (tp + fp + 1e-8)
    r_m = tp / (tp + fn + 1e-8)
    result["mBestF1"] = 2 * p_m * r_m / (p_m + r_m + 1e-8)

    return result


optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
best_eval_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss_sum, train_batches = 0.0, 0

    for batch in tqdm(train_loader, desc=f"[{epoch}/{EPOCHS}] train", leave=False):
        channel_ids   = batch["channel_ids"].to(device)
        telop_mask    = batch["telop_mask"].to(device)
        time_feats    = batch["time_feats"].to(device)
        slot_bbox     = batch["slot_bbox"].to(device)
        frame_mask    = batch["frame_mask"].to(device)
        slot_inst_idx = batch["slot_inst_idx"].to(device)
        inst_diff_ch  = batch["inst_diff_ch"].to(device)
        inst_diff_vid = batch["inst_diff_vid"].to(device)
        inst_diff_stt = batch["inst_diff_stt"].to(device)
        inst_scalars  = batch["inst_scalars"].to(device)

        active_mask = telop_mask & frame_mask.unsqueeze(-1)

        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
            x_logits, y_logits = model(
                channel_ids, telop_mask, time_feats, frame_mask,
                slot_inst_idx, inst_diff_ch, inst_diff_vid, inst_diff_stt, inst_scalars,
            )
            loss = slot_xy_loss(x_logits, y_logits, slot_bbox, active_mask)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_loss_sum += loss.item()
        train_batches += 1

    model.eval()
    eval_loss_sum, eval_batches = 0.0, 0
    all_metrics = []

    with torch.no_grad():
        for batch in eval_loader:
            channel_ids   = batch["channel_ids"].to(device)
            telop_mask    = batch["telop_mask"].to(device)
            time_feats    = batch["time_feats"].to(device)
            slot_bbox     = batch["slot_bbox"].to(device)
            frame_mask    = batch["frame_mask"].to(device)
            slot_inst_idx = batch["slot_inst_idx"].to(device)
            inst_diff_ch  = batch["inst_diff_ch"].to(device)
            inst_diff_vid = batch["inst_diff_vid"].to(device)
            inst_diff_stt = batch["inst_diff_stt"].to(device)
            inst_scalars  = batch["inst_scalars"].to(device)

            active_mask = telop_mask & frame_mask.unsqueeze(-1)

            with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
                x_logits, y_logits = model(
                    channel_ids, telop_mask, time_feats, frame_mask,
                    slot_inst_idx, inst_diff_ch, inst_diff_vid, inst_diff_stt, inst_scalars,
                )
                loss = slot_xy_loss(x_logits, y_logits, slot_bbox, active_mask)

            eval_loss_sum += loss.item()
            eval_batches += 1

            m = compute_slot_metrics(
                x_logits, y_logits, telop_mask, slot_bbox, frame_mask,
            )
            if m is not None:
                all_metrics.append(m)

    scheduler.step()

    train_loss = train_loss_sum / max(train_batches, 1)
    eval_loss  = eval_loss_sum  / max(eval_batches, 1)
    lr_now = optimizer.param_groups[0]["lr"]

    if all_metrics:
        avg_m = {k: np.mean([m[k] for m in all_metrics])
                 for k in all_metrics[0] if "Th" not in k}
        avg_thx = np.mean([m["bestThX"] for m in all_metrics])
        avg_thy = np.mean([m["bestThY"] for m in all_metrics])
    else:
        avg_m = {"P": 0, "R": 0, "F1": 0, "bestF1": 0, "hungF1": 0,
                 "mBestF1": 0, "xF1": 0, "yF1": 0}
        avg_thx, avg_thy = 0.5, 0.5

    print(
        f"[{epoch:>3}/{EPOCHS}]  "
        f"train={train_loss:.4f}  eval={eval_loss:.4f}  "
        f"P={avg_m.get('P',0):.3f}  R={avg_m.get('R',0):.3f}  "
        f"F1={avg_m.get('F1',0):.3f}  best={avg_m.get('bestF1',0):.3f}  "
        f"th=({avg_thx:.2f},{avg_thy:.2f})  "
        f"hF1={avg_m.get('hungF1',0):.3f}  "
        f"mF1={avg_m.get('mBestF1',0):.3f}  "
        f"xF1={avg_m.get('xF1',0):.3f}  yF1={avg_m.get('yF1',0):.3f}  "
        f"lr={lr_now:.2e}"
    )

    ckpt = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "eval_loss": eval_loss,
        "eval_metrics": avg_m,
        "channel2id": channel2id,
    }
    torch.save(ckpt, os.path.join(SAVE_DIR, f"epoch_{epoch:03d}.pt"))

    if eval_loss < best_eval_loss:
        best_eval_loss = eval_loss
        torch.save(ckpt, os.path.join(SAVE_DIR, "best.pt"))
        print(f"   💾 best 갱신 (eval_loss={eval_loss:.4f})")

print(f"\n✅ 완료. Best eval_loss={best_eval_loss:.4f}")